In [ ]:
# ============================================
# CELL 1/3: build GRAD_MAP  (|g|)
# ============================================

import os, sys, json, time
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import torch

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# -----------------------------
# CONFIG
# -----------------------------
RUN_DIR = "out/E4_clean_2k_adamw"
ITER = 800

K_BATCHES = 1024
SEED = 123

COORD_BUDGET_PER_GROUP = 600_000
N_COORD_SEEDS = 3
SAMPLES_PER_GROUP = 5_000_000

FIG_DIRNAME = "figures_tempered"
OVERWRITE = False
COMPRESS = True

ENABLE_TF32 = False
USE_CUDNN_BENCHMARK = True

UPDATE_EVERY = 8
PRINT_EVERY = 64
# -----------------------------

run_dir = Path(RUN_DIR).resolve()
assert run_dir.exists(), f"RUN_DIR not found: {run_dir}"
fig_dir = run_dir / FIG_DIRNAME
fig_dir.mkdir(parents=True, exist_ok=True)

npz_path  = fig_dir / f"grad_map_iter{ITER:07d}.npz"
meta_path = fig_dir / f"grad_map_iter{ITER:07d}.meta.json"

print("[info] saving:", npz_path)

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
PHASE_SEED = {"early": 111, "mid": 222, "late": 333}

if npz_path.exists() and not OVERWRITE:
    print("[ok] exists, loading")
    z = np.load(npz_path, allow_pickle=False)
    grad_map = {k: np.asarray(z[k]) for k in z.files}
    print("[ok] loaded groups:", len(grad_map))
else:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this cell.")
    DEVICE = "cuda"
    torch.backends.cuda.matmul.allow_tf32 = bool(ENABLE_TF32)
    torch.backends.cudnn.allow_tf32 = bool(ENABLE_TF32)
    torch.set_float32_matmul_precision("highest" if not ENABLE_TF32 else "high")
    torch.backends.cudnn.benchmark = bool(USE_CUDNN_BENCHMARK)

    sys.path.insert(0, str(Path.cwd()))
    from model import GPT, GPTConfig

    cfg = json.load(open(run_dir / "config_resolved.json", "r", encoding="utf-8"))
    n_layer = int(cfg.get("n_layer", 12))
    n_head  = int(cfg.get("n_head", 12))
    n_embd  = int(cfg.get("n_embd", 768))
    block_size = int(cfg.get("block_size", 1024))
    bias = bool(cfg.get("bias", False))
    dropout = float(cfg.get("dropout", 0.0))
    batch_size = int(cfg.get("batch_size", 12))
    vocab_size = int(cfg.get("vocab_size", 50304))

    data_dir = Path(cfg.get("data_dir", "data/openwebtext"))
    data_dir = data_dir if data_dir.is_absolute() else (Path.cwd() / data_dir).resolve()

    model = GPT(GPTConfig(
        block_size=block_size, vocab_size=vocab_size,
        n_layer=n_layer, n_head=n_head, n_embd=n_embd,
        dropout=dropout, bias=bias,
    )).to(DEVICE)

    ckpt_path = run_dir / "checkpoints" / f"ckpt_iter{ITER:07d}.pt"
    assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    model.load_state_dict(ckpt["model"], strict=True)

    model = model.to(dtype=torch.float32)
    model.train()
    name2param = dict(model.named_parameters())

    train_bin = data_dir / "train.bin"
    assert train_bin.exists(), f"train.bin not found: {train_bin}"
    train_data = np.memmap(train_bin, dtype=np.uint16, mode="r")
    train_len = int(train_data.shape[0])
    assert train_len > block_size + 2

    def get_batch_fast(bs: int, T: int, device: str):
        ix = torch.randint(train_len - T - 1, (bs,), device="cpu")
        x = torch.stack([torch.from_numpy(train_data[i:i+T].astype(np.int64, copy=False)) for i in ix])
        y = torch.stack([torch.from_numpy(train_data[i+1:i+1+T].astype(np.int64, copy=False)) for i in ix])
        return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

    b0 = n_layer // 3
    b1 = 2 * n_layer // 3
    PHASE_LAYERS = {"early": list(range(0, b0)), "mid": list(range(b0, b1)), "late": list(range(b1, n_layer))}

    def ordered_groups():
        out = []
        for phase in ["early","mid","late"]:
            for comp in ["q","k","v","proj"]:
                out.append(f"attn_{phase}_{comp}")
            for comp in ["fc","proj"]:
                out.append(f"mlp_{phase}_{comp}")
        return out

    GROUPS = ordered_groups()

    def sample_indices(numel: int, m: int, rng_local: np.random.Generator) -> np.ndarray:
        return rng_local.choice(numel, size=min(m, numel), replace=False).astype(np.int64)

    def to_idx_t(idx_np: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(idx_np).to(device=DEVICE, dtype=torch.long)

    def build_attn_qkv(layers, block: str, total_samples: int, rng_local):
        assert block in ("q","k","v")
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.attn.c_attn.weight"
            p = name2param[pname]
            numel = int(p.numel())
            block_numel = numel // 3
            offset = {"q":0,"k":1,"v":2}[block] * block_numel
            idx = sample_indices(block_numel, per_layer, rng_local) + offset
            samps.append((pname, to_idx_t(idx)))
        return samps

    def build_param(layers, suffix, total_samples, rng_local):
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.{suffix}"
            p = name2param[pname]
            idx = sample_indices(int(p.numel()), per_layer, rng_local)
            samps.append((pname, to_idx_t(idx)))
        return samps

    group_samplers = {}
    for phase, layers in PHASE_LAYERS.items():
        for comp in ["q","k","v","proj"]:
            group_samplers[f"attn_{phase}_{comp}"] = []
        for comp in ["fc","proj"]:
            group_samplers[f"mlp_{phase}_{comp}"] = []
        for s in range(N_COORD_SEEDS):
            rng_s = np.random.default_rng(SEED + 10_000*s + PHASE_SEED[phase])
            group_samplers[f"attn_{phase}_q"].append(build_attn_qkv(layers, "q", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_k"].append(build_attn_qkv(layers, "k", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_v"].append(build_attn_qkv(layers, "v", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_proj"].append(build_param(layers, "attn.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_fc"].append(build_param(layers, "mlp.c_fc.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_proj"].append(build_param(layers, "mlp.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))

    used_param_names = sorted({pname for seedlist in group_samplers.values() for samps in seedlist for (pname, _) in samps})

    class Reservoir:
        def __init__(self, size: int, rng: np.random.Generator):
            self.size = int(size)
            self.rng = rng
            self.buf = np.empty((self.size,), dtype=np.float32)
            self.filled = 0
            self.seen = 0
            self.replaced = 0

        def update(self, x: np.ndarray):
            x = np.asarray(x, dtype=np.float32).reshape(-1)
            m = int(x.size)
            if m <= 0:
                return
            if self.filled < self.size:
                take = min(self.size - self.filled, m)
                self.buf[self.filled:self.filled+take] = x[:take]
                self.filled += take
                self.seen += take
                x = x[take:]
                m = int(x.size)
                if m <= 0:
                    return
            base = int(self.seen)
            idx = base + np.arange(1, m+1, dtype=np.int64)
            prob = self.size / idx.astype(np.float64)
            keep = (self.rng.random(m) < prob)
            nk = int(np.sum(keep))
            if nk > 0:
                repl = self.rng.integers(0, self.size, size=nk, endpoint=False)
                self.buf[repl] = x[keep]
                self.replaced += nk
            self.seen += m

        def to_array(self):
            return self.buf.copy() if self.filled >= self.size else self.buf[:self.filled].copy()

    reservoirs = {g: Reservoir(SAMPLES_PER_GROUP, np.random.default_rng(SEED + 999 + i)) for i, g in enumerate(GROUPS)}
    amp_ctx = lambda: nullcontext()

    it = range(K_BATCHES) if tqdm is None else tqdm(range(K_BATCHES), total=K_BATCHES, desc="batches |g|", dynamic_ncols=True)
    t0 = time.time()

    for k in it:
        model.zero_grad(set_to_none=True)
        X, Y = get_batch_fast(batch_size, block_size, DEVICE)
        with amp_ctx():
            _, loss = model(X, Y)
        loss.backward()

        flat_cache = {}
        for pname in used_param_names:
            gg = name2param[pname].grad
            flat_cache[pname] = None if gg is None else gg.detach().flatten()

        for gname in GROUPS:
            for sidx, samplers in enumerate(group_samplers[gname]):
                chunks = []
                for pname, idx_t in samplers:
                    flat = flat_cache[pname]
                    if flat is None:
                        chunks.append(torch.zeros((idx_t.numel(),), device=DEVICE, dtype=torch.float32))
                    else:
                        chunks.append(flat.index_select(0, idx_t))
                vec = torch.cat(chunks, dim=0)
                reservoirs[gname].update(vec.abs_().detach().cpu().numpy().astype(np.float32, copy=False))

        if tqdm is not None and (k % UPDATE_EVERY == 0 or k == K_BATCHES-1):
            fills = np.array([reservoirs[g].filled / SAMPLES_PER_GROUP for g in GROUPS])
            it.set_postfix(min=f"{fills.min()*100:4.1f}%", avg=f"{fills.mean()*100:4.1f}%")

        if (k+1) % PRINT_EVERY == 0:
            avg_seen = float(np.mean([reservoirs[g].seen for g in GROUPS]))
            print(f"[detail] batch {k+1}/{K_BATCHES}, avg_seen={avg_seen/1e6:.2f}M")

    grad_map = {g: reservoirs[g].to_array() for g in GROUPS}
    if COMPRESS:
        np.savez_compressed(str(npz_path), **grad_map)
    else:
        np.savez(str(npz_path), **grad_map)

    meta = dict(kind="grad_map", run_dir=str(run_dir), iter=int(ITER), seed=int(SEED),
                k_batches=int(K_BATCHES), samples_per_group=int(SAMPLES_PER_GROUP),
                coord_budget_per_group=int(COORD_BUDGET_PER_GROUP), n_coord_seeds=int(N_COORD_SEEDS),
                enable_tf32=bool(ENABLE_TF32), npz=str(npz_path))
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[saved]", npz_path, meta_path)

grad_map

1
[info] saving: /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/grad_map_iter0000800.npz
[ok] exists, loading
[ok] loaded groups: 18


{'attn_early_q': array([4.0355790e-05, 1.4245138e-06, 5.6500867e-05, ..., 7.2304014e-05,
        3.7369420e-05, 7.9539905e-07], shape=(5000000,), dtype=float32),
 'attn_early_k': array([6.4203778e-07, 3.5436926e-06, 2.4780778e-05, ..., 4.2966658e-05,
        4.2291584e-05, 4.4154508e-06], shape=(5000000,), dtype=float32),
 'attn_early_v': array([4.2509116e-04, 6.7900663e-05, 1.5936284e-04, ..., 9.0229141e-06,
        1.4237560e-05, 4.1051433e-05], shape=(5000000,), dtype=float32),
 'attn_early_proj': array([2.1519627e-05, 7.5542768e-05, 6.3024032e-05, ..., 3.4352648e-05,
        1.6333133e-06, 1.3276091e-04], shape=(5000000,), dtype=float32),
 'mlp_early_fc': array([3.2988220e-05, 6.5460035e-06, 1.1343942e-04, ..., 1.0041964e-05,
        3.4768782e-05, 5.4749571e-06], shape=(5000000,), dtype=float32),
 'mlp_early_proj': array([3.1787164e-05, 5.0628256e-05, 3.1837876e-04, ..., 6.9849157e-05,
        2.3367646e-04, 5.7616715e-05], shape=(5000000,), dtype=float32),
 'attn_mid_q': array([1

In [2]:
# ============================================
# CELL 2/3: build DELTA_MAP (rolling |g_t - g_{t-1}|)
# ============================================

import os, sys, json, time
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import torch

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# -----------------------------
# CONFIG
# -----------------------------
RUN_DIR = "out/E4_clean_2k_adamw"
ITER = 800

K_BATCHES = 1024
SEED = 123

COORD_BUDGET_PER_GROUP = 600_000
N_COORD_SEEDS = 3
SAMPLES_PER_GROUP = 5_000_000

FIG_DIRNAME = "figures_tempered"
OVERWRITE = False
COMPRESS = True

ENABLE_TF32 = False
USE_CUDNN_BENCHMARK = True

UPDATE_EVERY = 8
PRINT_EVERY = 64
# -----------------------------

run_dir = Path(RUN_DIR).resolve()
assert run_dir.exists(), f"RUN_DIR not found: {run_dir}"
fig_dir = run_dir / FIG_DIRNAME
fig_dir.mkdir(parents=True, exist_ok=True)

npz_path  = fig_dir / f"delta_map_iter{ITER:07d}.npz"
meta_path = fig_dir / f"delta_map_iter{ITER:07d}.meta.json"

print("[info] saving:", npz_path)

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
PHASE_SEED = {"early": 111, "mid": 222, "late": 333}

if npz_path.exists() and not OVERWRITE:
    print("[ok] exists, loading")
    z = np.load(npz_path, allow_pickle=False)
    delta_map = {k: np.asarray(z[k]) for k in z.files}
    print("[ok] loaded groups:", len(delta_map))
else:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this cell.")
    DEVICE = "cuda"
    torch.backends.cuda.matmul.allow_tf32 = bool(ENABLE_TF32)
    torch.backends.cudnn.allow_tf32 = bool(ENABLE_TF32)
    torch.set_float32_matmul_precision("highest" if not ENABLE_TF32 else "high")
    torch.backends.cudnn.benchmark = bool(USE_CUDNN_BENCHMARK)

    sys.path.insert(0, str(Path.cwd()))
    from model import GPT, GPTConfig

    cfg = json.load(open(run_dir / "config_resolved.json", "r", encoding="utf-8"))
    n_layer = int(cfg.get("n_layer", 12))
    n_head  = int(cfg.get("n_head", 12))
    n_embd  = int(cfg.get("n_embd", 768))
    block_size = int(cfg.get("block_size", 1024))
    bias = bool(cfg.get("bias", False))
    dropout = float(cfg.get("dropout", 0.0))
    batch_size = int(cfg.get("batch_size", 12))
    vocab_size = int(cfg.get("vocab_size", 50304))

    data_dir = Path(cfg.get("data_dir", "data/openwebtext"))
    data_dir = data_dir if data_dir.is_absolute() else (Path.cwd() / data_dir).resolve()

    model = GPT(GPTConfig(
        block_size=block_size, vocab_size=vocab_size,
        n_layer=n_layer, n_head=n_head, n_embd=n_embd,
        dropout=dropout, bias=bias,
    )).to(DEVICE)

    ckpt_path = run_dir / "checkpoints" / f"ckpt_iter{ITER:07d}.pt"
    assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    model.load_state_dict(ckpt["model"], strict=True)

    model = model.to(dtype=torch.float32)
    model.train()
    name2param = dict(model.named_parameters())

    train_bin = data_dir / "train.bin"
    assert train_bin.exists(), f"train.bin not found: {train_bin}"
    train_data = np.memmap(train_bin, dtype=np.uint16, mode="r")
    train_len = int(train_data.shape[0])
    assert train_len > block_size + 2

    def get_batch_fast(bs: int, T: int, device: str):
        ix = torch.randint(train_len - T - 1, (bs,), device="cpu")
        x = torch.stack([torch.from_numpy(train_data[i:i+T].astype(np.int64, copy=False)) for i in ix])
        y = torch.stack([torch.from_numpy(train_data[i+1:i+1+T].astype(np.int64, copy=False)) for i in ix])
        return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

    b0 = n_layer // 3
    b1 = 2 * n_layer // 3
    PHASE_LAYERS = {"early": list(range(0, b0)), "mid": list(range(b0, b1)), "late": list(range(b1, n_layer))}

    def ordered_groups():
        out = []
        for phase in ["early","mid","late"]:
            for comp in ["q","k","v","proj"]:
                out.append(f"attn_{phase}_{comp}")
            for comp in ["fc","proj"]:
                out.append(f"mlp_{phase}_{comp}")
        return out
    GROUPS = ordered_groups()

    def sample_indices(numel: int, m: int, rng_local: np.random.Generator) -> np.ndarray:
        return rng_local.choice(numel, size=min(m, numel), replace=False).astype(np.int64)

    def to_idx_t(idx_np: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(idx_np).to(device=DEVICE, dtype=torch.long)

    def build_attn_qkv(layers, block: str, total_samples: int, rng_local):
        assert block in ("q","k","v")
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.attn.c_attn.weight"
            p = name2param[pname]
            numel = int(p.numel())
            block_numel = numel // 3
            offset = {"q":0,"k":1,"v":2}[block] * block_numel
            idx = sample_indices(block_numel, per_layer, rng_local) + offset
            samps.append((pname, to_idx_t(idx)))
        return samps

    def build_param(layers, suffix, total_samples, rng_local):
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.{suffix}"
            p = name2param[pname]
            idx = sample_indices(int(p.numel()), per_layer, rng_local)
            samps.append((pname, to_idx_t(idx)))
        return samps

    group_samplers = {}
    for phase, layers in PHASE_LAYERS.items():
        for comp in ["q","k","v","proj"]:
            group_samplers[f"attn_{phase}_{comp}"] = []
        for comp in ["fc","proj"]:
            group_samplers[f"mlp_{phase}_{comp}"] = []
        for s in range(N_COORD_SEEDS):
            rng_s = np.random.default_rng(SEED + 10_000*s + PHASE_SEED[phase])
            group_samplers[f"attn_{phase}_q"].append(build_attn_qkv(layers, "q", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_k"].append(build_attn_qkv(layers, "k", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_v"].append(build_attn_qkv(layers, "v", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_proj"].append(build_param(layers, "attn.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_fc"].append(build_param(layers, "mlp.c_fc.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_proj"].append(build_param(layers, "mlp.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))

    used_param_names = sorted({pname for seedlist in group_samplers.values() for samps in seedlist for (pname, _) in samps})

    class Reservoir:
        def __init__(self, size: int, rng: np.random.Generator):
            self.size = int(size)
            self.rng = rng
            self.buf = np.empty((self.size,), dtype=np.float32)
            self.filled = 0
            self.seen = 0
            self.replaced = 0

        def update(self, x: np.ndarray):
            x = np.asarray(x, dtype=np.float32).reshape(-1)
            m = int(x.size)
            if m <= 0:
                return
            if self.filled < self.size:
                take = min(self.size - self.filled, m)
                self.buf[self.filled:self.filled+take] = x[:take]
                self.filled += take
                self.seen += take
                x = x[take:]
                m = int(x.size)
                if m <= 0:
                    return
            base = int(self.seen)
            idx = base + np.arange(1, m+1, dtype=np.int64)
            prob = self.size / idx.astype(np.float64)
            keep = (self.rng.random(m) < prob)
            nk = int(np.sum(keep))
            if nk > 0:
                repl = self.rng.integers(0, self.size, size=nk, endpoint=False)
                self.buf[repl] = x[keep]
                self.replaced += nk
            self.seen += m

        def to_array(self):
            return self.buf.copy() if self.filled >= self.size else self.buf[:self.filled].copy()

    reservoirs = {g: Reservoir(SAMPLES_PER_GROUP, np.random.default_rng(SEED + 999 + i)) for i, g in enumerate(GROUPS)}
    prev = {g: [None]*N_COORD_SEEDS for g in GROUPS}
    amp_ctx = lambda: nullcontext()

    it = range(K_BATCHES) if tqdm is None else tqdm(range(K_BATCHES), total=K_BATCHES, desc="batches |g_t-g_{t-1}|", dynamic_ncols=True)

    for k in it:
        model.zero_grad(set_to_none=True)
        X, Y = get_batch_fast(batch_size, block_size, DEVICE)
        with amp_ctx():
            _, loss = model(X, Y)
        loss.backward()

        flat_cache = {}
        for pname in used_param_names:
            gg = name2param[pname].grad
            flat_cache[pname] = None if gg is None else gg.detach().flatten()

        for gname in GROUPS:
            for sidx, samplers in enumerate(group_samplers[gname]):
                chunks = []
                for pname, idx_t in samplers:
                    flat = flat_cache[pname]
                    if flat is None:
                        chunks.append(torch.zeros((idx_t.numel(),), device=DEVICE, dtype=torch.float32))
                    else:
                        chunks.append(flat.index_select(0, idx_t))
                vec = torch.cat(chunks, dim=0)

                if prev[gname][sidx] is None:
                    prev[gname][sidx] = vec
                    continue
                d = (prev[gname][sidx] - vec).abs_()
                prev[gname][sidx] = vec
                reservoirs[gname].update(d.detach().cpu().numpy().astype(np.float32, copy=False))

        if tqdm is not None and (k % UPDATE_EVERY == 0 or k == K_BATCHES-1):
            fills = np.array([reservoirs[g].filled / SAMPLES_PER_GROUP for g in GROUPS])
            it.set_postfix(min=f"{fills.min()*100:4.1f}%", avg=f"{fills.mean()*100:4.1f}%")

    delta_map = {g: reservoirs[g].to_array() for g in GROUPS}
    if COMPRESS:
        np.savez_compressed(str(npz_path), **delta_map)
    else:
        np.savez(str(npz_path), **delta_map)

    meta = dict(kind="delta_map", run_dir=str(run_dir), iter=int(ITER), seed=int(SEED),
                k_batches=int(K_BATCHES), samples_per_group=int(SAMPLES_PER_GROUP),
                coord_budget_per_group=int(COORD_BUDGET_PER_GROUP), n_coord_seeds=int(N_COORD_SEEDS),
                enable_tf32=bool(ENABLE_TF32), npz=str(npz_path))
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[saved]", npz_path, meta_path)

delta_map

[info] saving: /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/delta_map_iter0000800.npz
number of parameters: 123.59M


/home/coder/tmp/ipykernel_516434/4016379568.py:93: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
batches |g_t-g_{t-1}|

[saved] /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/delta_map_iter0000800.npz /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/delta_map_iter0000800.meta.json


{'attn_early_q': array([3.7438207e-05, 1.8783112e-06, 4.9227034e-05, ..., 5.6127130e-05,
        9.0128975e-05, 8.7385088e-06], shape=(5000000,), dtype=float32),
 'attn_early_k': array([2.5747639e-05, 1.9817398e-05, 9.1841703e-06, ..., 2.8230235e-05,
        2.9627774e-05, 2.5052652e-05], shape=(5000000,), dtype=float32),
 'attn_early_v': array([1.0744319e-05, 3.9454022e-05, 1.5500924e-04, ..., 5.6956222e-05,
        1.9732232e-05, 4.3812670e-05], shape=(5000000,), dtype=float32),
 'attn_early_proj': array([7.1186114e-06, 1.1786477e-04, 4.1689760e-05, ..., 5.1848016e-05,
        1.4588534e-04, 6.7607139e-04], shape=(5000000,), dtype=float32),
 'mlp_early_fc': array([3.3651893e-05, 2.2530508e-05, 1.0529917e-04, ..., 3.4195386e-05,
        2.1944779e-05, 3.1825391e-05], shape=(5000000,), dtype=float32),
 'mlp_early_proj': array([1.9971306e-04, 1.2335167e-04, 3.2489907e-04, ..., 6.2646854e-05,
        3.6058947e-05, 1.2893935e-04], shape=(5000000,), dtype=float32),
 'attn_mid_q': array([2

In [3]:
# ============================================
# CELL 3/3: build NOISE_MAP (noise-only |g_a - g_b| at fixed w)
# ============================================

import os, sys, json, time
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import torch

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# -----------------------------
# CONFIG
# -----------------------------
RUN_DIR = "out/E4_clean_2k_adamw"
ITER = 800

K_PAIRS = 512       # 1 pair = 2 backward passes
SEED = 123

COORD_BUDGET_PER_GROUP = 600_000
N_COORD_SEEDS = 3
SAMPLES_PER_GROUP = 5_000_000

FIG_DIRNAME = "figures_tempered"
OVERWRITE = False
COMPRESS = True

ENABLE_TF32 = False
USE_CUDNN_BENCHMARK = True

UPDATE_EVERY = 4
PRINT_EVERY = 32
# -----------------------------

run_dir = Path(RUN_DIR).resolve()
assert run_dir.exists(), f"RUN_DIR not found: {run_dir}"
fig_dir = run_dir / FIG_DIRNAME
fig_dir.mkdir(parents=True, exist_ok=True)

npz_path  = fig_dir / f"noise_map_iter{ITER:07d}.npz"
meta_path = fig_dir / f"noise_map_iter{ITER:07d}.meta.json"

print("[info] saving:", npz_path)

torch.manual_seed(SEED)
np.random.seed(SEED)
PHASE_SEED = {"early": 111, "mid": 222, "late": 333}

if npz_path.exists() and not OVERWRITE:
    print("[ok] exists, loading")
    z = np.load(npz_path, allow_pickle=False)
    noise_map = {k: np.asarray(z[k]) for k in z.files}
    print("[ok] loaded groups:", len(noise_map))
else:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this cell.")
    DEVICE = "cuda"
    torch.backends.cuda.matmul.allow_tf32 = bool(ENABLE_TF32)
    torch.backends.cudnn.allow_tf32 = bool(ENABLE_TF32)
    torch.set_float32_matmul_precision("highest" if not ENABLE_TF32 else "high")
    torch.backends.cudnn.benchmark = bool(USE_CUDNN_BENCHMARK)

    sys.path.insert(0, str(Path.cwd()))
    from model import GPT, GPTConfig

    cfg = json.load(open(run_dir / "config_resolved.json", "r", encoding="utf-8"))
    n_layer = int(cfg.get("n_layer", 12))
    n_head  = int(cfg.get("n_head", 12))
    n_embd  = int(cfg.get("n_embd", 768))
    block_size = int(cfg.get("block_size", 1024))
    bias = bool(cfg.get("bias", False))
    dropout = float(cfg.get("dropout", 0.0))
    batch_size = int(cfg.get("batch_size", 12))
    vocab_size = int(cfg.get("vocab_size", 50304))

    data_dir = Path(cfg.get("data_dir", "data/openwebtext"))
    data_dir = data_dir if data_dir.is_absolute() else (Path.cwd() / data_dir).resolve()

    model = GPT(GPTConfig(
        block_size=block_size, vocab_size=vocab_size,
        n_layer=n_layer, n_head=n_head, n_embd=n_embd,
        dropout=dropout, bias=bias,
    )).to(DEVICE)

    ckpt_path = run_dir / "checkpoints" / f"ckpt_iter{ITER:07d}.pt"
    assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    model.load_state_dict(ckpt["model"], strict=True)

    model = model.to(dtype=torch.float32)
    model.train()
    name2param = dict(model.named_parameters())

    train_bin = data_dir / "train.bin"
    assert train_bin.exists(), f"train.bin not found: {train_bin}"
    train_data = np.memmap(train_bin, dtype=np.uint16, mode="r")
    train_len = int(train_data.shape[0])
    assert train_len > block_size + 2

    def get_batch_fast(bs: int, T: int, device: str):
        ix = torch.randint(train_len - T - 1, (bs,), device="cpu")
        x = torch.stack([torch.from_numpy(train_data[i:i+T].astype(np.int64, copy=False)) for i in ix])
        y = torch.stack([torch.from_numpy(train_data[i+1:i+1+T].astype(np.int64, copy=False)) for i in ix])
        return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

    b0 = n_layer // 3
    b1 = 2 * n_layer // 3
    PHASE_LAYERS = {"early": list(range(0, b0)), "mid": list(range(b0, b1)), "late": list(range(b1, n_layer))}

    def ordered_groups():
        out = []
        for phase in ["early","mid","late"]:
            for comp in ["q","k","v","proj"]:
                out.append(f"attn_{phase}_{comp}")
            for comp in ["fc","proj"]:
                out.append(f"mlp_{phase}_{comp}")
        return out
    GROUPS = ordered_groups()

    def sample_indices(numel: int, m: int, rng_local: np.random.Generator) -> np.ndarray:
        return rng_local.choice(numel, size=min(m, numel), replace=False).astype(np.int64)

    def to_idx_t(idx_np: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(idx_np).to(device=DEVICE, dtype=torch.long)

    def build_attn_qkv(layers, block: str, total_samples: int, rng_local):
        assert block in ("q","k","v")
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.attn.c_attn.weight"
            p = name2param[pname]
            numel = int(p.numel())
            block_numel = numel // 3
            offset = {"q":0,"k":1,"v":2}[block] * block_numel
            idx = sample_indices(block_numel, per_layer, rng_local) + offset
            samps.append((pname, to_idx_t(idx)))
        return samps

    def build_param(layers, suffix, total_samples, rng_local):
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.{suffix}"
            p = name2param[pname]
            idx = sample_indices(int(p.numel()), per_layer, rng_local)
            samps.append((pname, to_idx_t(idx)))
        return samps

    group_samplers = {}
    for phase, layers in PHASE_LAYERS.items():
        for comp in ["q","k","v","proj"]:
            group_samplers[f"attn_{phase}_{comp}"] = []
        for comp in ["fc","proj"]:
            group_samplers[f"mlp_{phase}_{comp}"] = []
        for s in range(N_COORD_SEEDS):
            rng_s = np.random.default_rng(SEED + 10_000*s + PHASE_SEED[phase])
            group_samplers[f"attn_{phase}_q"].append(build_attn_qkv(layers, "q", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_k"].append(build_attn_qkv(layers, "k", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_v"].append(build_attn_qkv(layers, "v", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_proj"].append(build_param(layers, "attn.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_fc"].append(build_param(layers, "mlp.c_fc.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_proj"].append(build_param(layers, "mlp.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))

    used_param_names = sorted({pname for seedlist in group_samplers.values() for samps in seedlist for (pname, _) in samps})

    class Reservoir:
        def __init__(self, size: int, rng: np.random.Generator):
            self.size = int(size)
            self.rng = rng
            self.buf = np.empty((self.size,), dtype=np.float32)
            self.filled = 0
            self.seen = 0
            self.replaced = 0

        def update(self, x: np.ndarray):
            x = np.asarray(x, dtype=np.float32).reshape(-1)
            m = int(x.size)
            if m <= 0:
                return
            if self.filled < self.size:
                take = min(self.size - self.filled, m)
                self.buf[self.filled:self.filled+take] = x[:take]
                self.filled += take
                self.seen += take
                x = x[take:]
                m = int(x.size)
                if m <= 0:
                    return
            base = int(self.seen)
            idx = base + np.arange(1, m+1, dtype=np.int64)
            prob = self.size / idx.astype(np.float64)
            keep = (self.rng.random(m) < prob)
            nk = int(np.sum(keep))
            if nk > 0:
                repl = self.rng.integers(0, self.size, size=nk, endpoint=False)
                self.buf[repl] = x[keep]
                self.replaced += nk
            self.seen += m

        def to_array(self):
            return self.buf.copy() if self.filled >= self.size else self.buf[:self.filled].copy()

    reservoirs = {g: Reservoir(SAMPLES_PER_GROUP, np.random.default_rng(SEED + 999 + i)) for i, g in enumerate(GROUPS)}
    amp_ctx = lambda: nullcontext()

    it = range(K_PAIRS) if tqdm is None else tqdm(range(K_PAIRS), total=K_PAIRS, desc="pairs |g_a-g_b|", dynamic_ncols=True)

    for k in it:
        # pass A
        model.zero_grad(set_to_none=True)
        Xa, Ya = get_batch_fast(batch_size, block_size, DEVICE)
        with amp_ctx():
            _, la = model(Xa, Ya)
        la.backward()

        flat_a = {}
        for pname in used_param_names:
            gg = name2param[pname].grad
            flat_a[pname] = None if gg is None else gg.detach().flatten()

        vecA_cpu = {}
        for gname in GROUPS:
            for sidx, samplers in enumerate(group_samplers[gname]):
                chunks = []
                for pname, idx_t in samplers:
                    flat = flat_a[pname]
                    if flat is None:
                        chunks.append(torch.zeros((idx_t.numel(),), device=DEVICE, dtype=torch.float32))
                    else:
                        chunks.append(flat.index_select(0, idx_t))
                vec = torch.cat(chunks, dim=0)
                vecA_cpu[(gname, sidx)] = vec.detach().cpu().numpy().astype(np.float32, copy=False)

        # pass B
        model.zero_grad(set_to_none=True)
        Xb, Yb = get_batch_fast(batch_size, block_size, DEVICE)
        with amp_ctx():
            _, lb = model(Xb, Yb)
        lb.backward()

        flat_b = {}
        for pname in used_param_names:
            gg = name2param[pname].grad
            flat_b[pname] = None if gg is None else gg.detach().flatten()

        for gname in GROUPS:
            for sidx, samplers in enumerate(group_samplers[gname]):
                chunks = []
                for pname, idx_t in samplers:
                    flat = flat_b[pname]
                    if flat is None:
                        chunks.append(torch.zeros((idx_t.numel(),), device=DEVICE, dtype=torch.float32))
                    else:
                        chunks.append(flat.index_select(0, idx_t))
                vecB = torch.cat(chunks, dim=0).detach().cpu().numpy().astype(np.float32, copy=False)
                d = np.abs(vecA_cpu[(gname, sidx)] - vecB)
                reservoirs[gname].update(d)

        if tqdm is not None and (k % UPDATE_EVERY == 0 or k == K_PAIRS-1):
            fills = np.array([reservoirs[g].filled / SAMPLES_PER_GROUP for g in GROUPS])
            it.set_postfix(min=f"{fills.min()*100:4.1f}%", avg=f"{fills.mean()*100:4.1f}%")

    noise_map = {g: reservoirs[g].to_array() for g in GROUPS}
    if COMPRESS:
        np.savez_compressed(str(npz_path), **noise_map)
    else:
        np.savez(str(npz_path), **noise_map)

    meta = dict(kind="noise_map", run_dir=str(run_dir), iter=int(ITER), seed=int(SEED),
                k_pairs=int(K_PAIRS), samples_per_group=int(SAMPLES_PER_GROUP),
                coord_budget_per_group=int(COORD_BUDGET_PER_GROUP), n_coord_seeds=int(N_COORD_SEEDS),
                enable_tf32=bool(ENABLE_TF32), npz=str(npz_path))
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[saved]", npz_path, meta_path)

noise_map

[info] saving: /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/noise_map_iter0000800.npz
[ok] exists, loading
[ok] loaded groups: 18


{'attn_early_q': array([3.1619013e-05, 4.4253279e-06, 1.2075907e-06, ..., 4.4819401e-05,
        6.1597448e-06, 8.5507427e-06], shape=(5000000,), dtype=float32),
 'attn_early_k': array([2.6771591e-05, 1.3493522e-05, 3.5186476e-06, ..., 8.1786049e-05,
        3.9850220e-05, 4.1836764e-05], shape=(5000000,), dtype=float32),
 'attn_early_v': array([6.7748886e-05, 2.1084725e-05, 3.1438911e-05, ..., 1.8246974e-05,
        3.6862119e-05, 1.1141453e-04], shape=(5000000,), dtype=float32),
 'attn_early_proj': array([2.54149199e-04, 7.88791513e-05, 1.09025816e-04, ...,
        1.74716290e-04, 1.34788861e-05, 1.97847607e-04],
       shape=(5000000,), dtype=float32),
 'mlp_early_fc': array([5.3834046e-06, 4.4174500e-05, 8.3762898e-06, ..., 3.9779123e-05,
        3.6363785e-05, 4.1560132e-05], shape=(5000000,), dtype=float32),
 'mlp_early_proj': array([6.3300045e-05, 2.8948860e-05, 3.7002901e-04, ..., 1.0866416e-04,
        1.4443757e-04, 6.4345222e-05], shape=(5000000,), dtype=float32),
 'attn_mid

In [9]:
# ============================================================
# FINAL FIGURE BUILDER (bright points + subsampling)
# Produces folder OUT_DIR with:
#   MAIN (noise): 3 PDFs
#   APPENDIX: full audit PDFs for noise/grad/delta
#
# Requires: grad_map, noise_map, delta_map in memory.
# ============================================================

import os, math
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from contextlib import contextmanager
from matplotlib.backends.backend_pdf import PdfPages

# -----------------------------
# USER SETTINGS
# -----------------------------
OUT_DIR = "paper_figures_uai"
EXPORT_PDFS = True
SHOW_PLOTS = False

MAKE_MAIN_NOISE = True
MAKE_MAIN_BEFF  = True

MAKE_APPENDIX_NOISE = True
MAKE_APPENDIX_GRAD  = True
MAKE_APPENDIX_DELTA = True

MAIN_REP_N = 6
ITER_TAG = globals().get("ITER", "?")

# Point styling (FIXED as requested)
DATA_MARKER_SIZE = 4.5
DATA_ALPHA = 0.85
DATA_EDGEWIDTH = 0.25
DATA_MAX_POINTS = 3500   # subsample scatter points per CCDF plot (keeps PDFs light)

MODEL_LINEWIDTH = 2.0

# -----------------------------
# FIT SETTINGS
# -----------------------------
Q_LO, Q_HI = 0.001, 0.99998
NBINS_CCDF = 800
NBINS_FIT  = 180
MIN_FIT_BINS = 60
BEFF_WINDOW = 21

CCDF_MAX_CANDIDATES = (0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001)
MIN_TAIL_COUNT_CANDIDATES = (300, 500, 1000, 2000, 5000, 10000, 20000)

BETA_GRID = np.unique(np.concatenate([np.linspace(0.2, 1.0, 17), np.array([0.15, 0.12, 0.10])]))
CUTOFF_EPS_EFFECT = 0.02

WEIGHT_TC_POWER   = 0.65
WEIGHT_CC_POWER   = 0.30
WEIGHT_CLIP_MAX   = 200_000.0

LEFT_K,  LEFT_BOOST  = 6,  2.0
RIGHT_K, RIGHT_BOOST = 18, 40.0

EPS = 1e-300
HOLDOUT_FRAC = 0.30
HOLDOUT_SEED = 777
MIN_SIGMA_LN = 1e-3

# -----------------------------
# Sanity: required maps
# -----------------------------
assert "noise_map" in globals() and isinstance(noise_map, dict) and len(noise_map) > 0, "noise_map missing"
assert "grad_map"  in globals() and isinstance(grad_map, dict)  and len(grad_map)  > 0, "grad_map missing"
assert "delta_map" in globals() and isinstance(delta_map, dict) and len(delta_map) > 0, "delta_map missing"

out_dir = Path(OUT_DIR).resolve()
out_dir.mkdir(parents=True, exist_ok=True)
print("[info] OUT_DIR:", out_dir)

@contextmanager
def _no_show():
    old = plt.show
    plt.show = lambda *a, **k: None
    try:
        yield
    finally:
        plt.show = old

def ordered_groups():
    out = []
    for phase in ["early", "mid", "late"]:
        for comp in ["q","k","v","proj"]:
            out.append(f"attn_{phase}_{comp}")
        for comp in ["fc","proj"]:
            out.append(f"mlp_{phase}_{comp}")
    return out

def _subsample_xy(x, y, max_points=DATA_MAX_POINTS):
    n = x.size
    if n <= max_points:
        return x, y
    idx = np.linspace(0, n-1, max_points).astype(np.int64)
    return x[idx], y[idx]

def _ccdf_logbins(x, q_lo=Q_LO, q_hi=Q_HI, nbins=NBINS_CCDF):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    x = x[x > 0]
    x.sort()
    n = x.size
    if n < 10:
        return np.array([]), np.array([]), np.array([])

    xmin = float(np.quantile(x, q_lo))
    xmax = float(np.quantile(x, q_hi))
    if not np.isfinite(xmin) or not np.isfinite(xmax) or xmax <= xmin:
        return np.array([]), np.array([]), np.array([])

    edges = np.logspace(np.log10(xmin), np.log10(xmax), nbins + 1)
    g = edges[1:]
    idx = np.searchsorted(x, g, side="right")
    cc = (n - idx) / n
    cc = np.clip(cc, 1.0/(n+1.0), 1.0)
    tc = n * cc

    keep = np.r_[True, np.diff(idx) != 0]
    return g[keep], cc[keep], tc[keep]

def _aggregate_by_logx_bins(x, ylog, tc, cc, nbins=NBINS_FIT):
    x = np.asarray(x, dtype=np.float64)
    ylog = np.asarray(ylog, dtype=np.float64)
    tc = np.asarray(tc, dtype=np.float64)
    cc = np.asarray(cc, dtype=np.float64)

    lx = np.log(x)
    lo, hi = float(lx.min()), float(lx.max())
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return x, ylog, np.ones_like(ylog)

    edges = np.linspace(lo, hi, nbins + 1)
    b = np.searchsorted(edges, lx, side="right") - 1
    b = np.clip(b, 0, nbins - 1)

    xs, ys, ws = [], [], []
    for bi in range(nbins):
        m = (b == bi)
        if not np.any(m):
            continue
        x_m = x[m]
        y_m = ylog[m]
        tc_m = tc[m]
        cc_m = cc[m]

        w = (np.clip(tc_m, 1.0, WEIGHT_CLIP_MAX) ** WEIGHT_TC_POWER) * (np.clip(cc_m, EPS, 1.0) ** (-WEIGHT_CC_POWER))
        wsum = float(np.sum(w))
        if wsum <= 0:
            continue

        x_bin = float(np.exp(np.sum(w * np.log(x_m)) / wsum))
        y_bin = float(np.sum(w * y_m) / wsum)

        xs.append(x_bin); ys.append(y_bin); ws.append(wsum)

    xs = np.asarray(xs); ys = np.asarray(ys); ws = np.asarray(ws)
    if xs.size:
        ws = ws.copy()
        ws[:min(LEFT_K, xs.size)] *= LEFT_BOOST
        ws[max(0, xs.size - RIGHT_K):] *= RIGHT_BOOST
        med = np.median(ws)
        if np.isfinite(med) and med > 0:
            ws /= med
        ws = np.clip(ws, 1e-12, 1e12)

    return xs, ys, ws

def _weighted_lstsq(A, y, w):
    w = np.asarray(w, dtype=np.float64)
    w = np.clip(w, 1e-12, None)
    sw = np.sqrt(w)
    beta, *_ = np.linalg.lstsq(A * sw[:, None], y * sw, rcond=None)
    return beta

def _aic_bic_from_sse(sse, n, k):
    sse = float(max(sse, 1e-300))
    n = int(n); k = int(k)
    return float(n * np.log(sse / n) + k * np.log(n))  # BIC

def _split_train_valid(n, seed=HOLDOUT_SEED, frac=HOLDOUT_FRAC):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    nv = max(1, int(np.floor(frac * n)))
    vidx = idx[:nv]
    tidx = idx[nv:]
    if tidx.size < 10:
        tidx = idx[nv//2:]
        vidx = idx[:nv//2]
    return tidx, vidx

def _beff_from_ccdf(g, cc, window=BEFF_WINDOW):
    logx = np.log(g)
    logy = np.log(np.clip(cc, EPS, 1.0))
    w = window // 2
    slopes = np.full_like(logx, np.nan, dtype=np.float64)
    for i in range(len(logx)):
        j0 = max(0, i - w)
        j1 = min(len(logx), i + w + 1)
        Xw = logx[j0:j1]; Yw = logy[j0:j1]
        if len(Xw) < 2:
            continue
        Xc = Xw - Xw.mean()
        denom = np.sum(Xc * Xc)
        if denom <= 0:
            continue
        slopes[i] = np.sum(Xc * (Yw - Yw.mean())) / denom
    return -slopes

# Models
def _log_ccdf_powerlaw(x, alpha, c0):
    x = np.asarray(x, dtype=np.float64)
    return c0 - alpha * np.log(np.clip(x, 1e-300, np.inf))

def _log_ccdf_tempered(x, alpha, lam, beta, c0):
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, 1e-300, np.inf)
    return c0 - alpha * np.log(x) - (x / lam) ** beta

_erfc = np.vectorize(math.erfc, otypes=[np.float64])
def _log_ccdf_lognormal(x, mu, sigma):
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, 1e-300, np.inf)
    sigma = float(max(sigma, MIN_SIGMA_LN))
    z = (np.log(x) - mu) / (sigma * np.sqrt(2.0))
    cc = 0.5 * _erfc(z)
    return np.log(np.clip(cc, EPS, 1.0))

def _k_params(model):
    return {"powerlaw": 2, "lognormal": 2, "tempered": 4}[model]

def _eval_sse(x, ylog, w, fit):
    m = fit["model"]
    if m == "powerlaw":
        yhat = _log_ccdf_powerlaw(x, fit["alpha"], fit["c0"])
    elif m == "lognormal":
        yhat = _log_ccdf_lognormal(x, fit["mu"], fit["sigma"])
    else:
        yhat = _log_ccdf_tempered(x, fit["alpha"], fit["lam"], fit["beta"], fit["c0"])
    return float(np.sum(w * (ylog - yhat) ** 2))

def _fit_powerlaw(x, ylog, w):
    X1 = np.log(x)
    A = np.stack([np.ones_like(X1), X1], axis=1)
    c0_hat, b1 = _weighted_lstsq(A, ylog, w)
    return dict(model="powerlaw", alpha=float(-b1), c0=float(c0_hat))

def _fit_lognormal(x, ylog, w):
    lx = np.log(x)
    sig_grid = np.linspace(0.2, 3.0, 60)
    best = None
    mu0 = float(np.sum(w * lx) / np.sum(w))
    for sigma in sig_grid:
        sigma = float(sigma)
        mu_grid = np.linspace(mu0 - 3*sigma, mu0 + 3*sigma, 61)
        for mu in mu_grid:
            fit = dict(model="lognormal", mu=float(mu), sigma=float(sigma))
            sse = _eval_sse(x, ylog, w, fit)
            if best is None or sse < best[0]:
                best = (sse, fit)
    return best[1]

def _fit_tempered_grid(x, ylog, w):
    best = None
    X1 = np.log(x)
    x_ref = float(np.exp(np.mean(X1)))
    for beta in BETA_GRID:
        beta = float(beta)
        u = (x / x_ref) ** beta
        A = np.stack([np.ones_like(X1), X1, u], axis=1)
        c0_hat, b1, b2 = _weighted_lstsq(A, ylog, w)
        if not np.isfinite(b2) or b2 >= 0:
            continue
        alpha = float(-b1)
        if not (np.isfinite(alpha) and alpha > 0):
            continue
        Acoef = float(-b2)
        lam = float(x_ref / (Acoef ** (1.0 / beta)))
        if not (np.isfinite(lam) and lam > 0):
            continue
        tmax = float((x.max() / lam) ** beta)
        if tmax < CUTOFF_EPS_EFFECT:
            continue
        fit = dict(model="tempered", alpha=alpha, beta=beta, lam=lam, c0=float(c0_hat))
        sse = _eval_sse(x, ylog, w, fit)
        if best is None or sse < best[0]:
            best = (sse, fit)
    return None if best is None else best[1]

def select_model_for_array(arr):
    arr = np.asarray(arr, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    arr = arr[arr > 0]
    if arr.size < 200_000:
        return None

    g_all, cc_all, tc_all = _ccdf_logbins(arr)
    if g_all.size < 200:
        return None

    best_global = None

    for ccdf_max in CCDF_MAX_CANDIDATES:
        for min_tail in MIN_TAIL_COUNT_CANDIDATES:
            fitmask = (cc_all <= ccdf_max) & (tc_all >= min_tail)
            if int(fitmask.sum()) < max(80, NBINS_FIT//2):
                continue

            g_fit = g_all[fitmask]
            cc_fit = cc_all[fitmask]
            tc_fit = tc_all[fitmask]
            ylog = np.log(np.clip(cc_fit, EPS, 1.0))

            xA, yA, wA = _aggregate_by_logx_bins(g_fit, ylog, tc_fit, cc_fit, nbins=NBINS_FIT)
            if xA.size < MIN_FIT_BINS:
                continue

            tidx, vidx = _split_train_valid(xA.size)
            xT, yT, wT = xA[tidx], yA[tidx], wA[tidx]
            xV, yV, wV = xA[vidx], yA[vidx], wA[vidx]

            fits = []
            f_pl = _fit_powerlaw(xT, yT, wT); fits.append(f_pl)
            f_ln = _fit_lognormal(xT, yT, wT); fits.append(f_ln)
            f_tp = _fit_tempered_grid(xT, yT, wT)
            if f_tp is not None: fits.append(f_tp)

            scored = []
            for f in fits:
                sseT = _eval_sse(xT, yT, wT, f)
                sseV = _eval_sse(xV, yV, wV, f)
                bic = _aic_bic_from_sse(sseT, n=len(xT), k=_k_params(f["model"]))
                scored.append(dict(fit=f, sse_train=sseT, sse_valid=sseV, bic=bic))
            scored.sort(key=lambda d: d["bic"])

            winner = scored[0]
            runner = scored[1] if len(scored) > 1 else None
            dBIC = (runner["bic"] - winner["bic"]) if runner is not None else np.nan

            tau = np.nan
            ident = ""
            if winner["fit"]["model"] == "tempered":
                lam = winner["fit"]["lam"]; beta = winner["fit"]["beta"]
                xmax = float(xA.max())
                tau = float((xmax/lam)**beta) if (np.isfinite(lam) and lam>0) else np.nan
                if (beta < 0.2) or (not np.isfinite(tau)) or (tau < 0.2) or (tau > 50):
                    ident = "lambda weakly identifiable"
            hold_ratio = float(winner["sse_valid"] / max(winner["sse_train"], 1e-300))

            key = (winner["bic"], winner["sse_valid"])
            pack = dict(
                g=g_all, cc=cc_all, tc=tc_all,
                fitmask=fitmask, ccdf_max=float(ccdf_max), min_tail=int(min_tail),
                winner=winner, runner=runner, dBIC=float(dBIC) if np.isfinite(dBIC) else np.nan,
                tau=tau, ident=ident, hold_ratio=hold_ratio,
            )
            if best_global is None or key < best_global["key"]:
                best_global = dict(key=key, sel=pack)

    return None if best_global is None else best_global["sel"]

# Plot formatting
def _tail_zoom_limits(g, fitmask, ccdf_max):
    x_min_fit = float(g[fitmask].min())
    x_max_fit = float(g[fitmask].max())
    y_top = min(1.0, ccdf_max * 1.2)
    y_bot = max(1e-9, ccdf_max / 50.0)
    return x_min_fit/3.0, x_max_fit*1.25, y_bot, y_top

def _ccdf_model_curve(g, fit):
    if fit["model"] == "powerlaw":
        return np.exp(np.clip(_log_ccdf_powerlaw(g, fit["alpha"], fit["c0"]), -745, 0.0))
    if fit["model"] == "lognormal":
        return np.exp(np.clip(_log_ccdf_lognormal(g, fit["mu"], fit["sigma"]), -745, 0.0))
    return np.exp(np.clip(_log_ccdf_tempered(g, fit["alpha"], fit["lam"], fit["beta"], fit["c0"]), -745, 0.0))

def _legend_outside(ax):
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0.0, fontsize=8, frameon=True)

def _fmt_fit_short(fit, tau=np.nan):
    if fit["model"] == "powerlaw":
        return f"PL alpha={fit['alpha']:.2f}"
    if fit["model"] == "lognormal":
        return f"LN sigma={fit['sigma']:.2f}"
    tau_str = "nan" if not np.isfinite(tau) else f"{tau:.2g}"
    return f"TP alpha={fit['alpha']:.2f}, beta={fit['beta']:.2f}, tau={tau_str}"

def _diag_box(sel):
    w = sel["winner"]["fit"]
    r = sel["runner"]["fit"] if sel["runner"] is not None else None
    lines = [
        f"winner: {w['model']}   runner: {(r['model'] if r else '—')}",
        f"BIC(w)={sel['winner']['bic']:.1f}  BIC(r)={(sel['runner']['bic'] if sel['runner'] else np.nan):.1f}  dBIC={sel['dBIC']:.1f}",
        f"holdout ratio (valid/train SSE)={sel['hold_ratio']:.2f}",
        f"window: CCDF<={sel['ccdf_max']}  tail_count>={sel['min_tail']}",
    ]
    if w["model"] == "tempered":
        tau_str = "nan" if not np.isfinite(sel["tau"]) else f"{sel['tau']:.3g}"
        lines.append(f"tau=(x_max/lambda)^beta={tau_str}  {sel['ident']}")
    return "\n".join(lines)

# -----------------------------
# Figure builders (main + appendix)
# -----------------------------
def fig_model_map(rows18, title, x_label, tag):
    enc = {"powerlaw":0, "lognormal":1, "tempered":2}
    M = np.array([enc[r["model"]] for r in rows18], dtype=np.int64).reshape(3, 6)
    D = np.array([r["dBIC"] for r in rows18], dtype=np.float64).reshape(3, 6)

    fig = plt.figure(figsize=(10.6, 4.3))
    ax = plt.gca()
    ax.imshow(M, aspect="auto")

    for i in range(3):
        for j in range(6):
            mname = ["PL","LN","TP"][M[i,j]]
            dv = D[i,j]
            s = "—" if not np.isfinite(dv) else f"{dv:.0f}"
            ax.text(j, i, f"{mname}\nd{s}", ha="center", va="center", fontsize=8)

    ax.set_yticks([0,1,2]); ax.set_yticklabels(["early","mid","late"])
    ax.set_xticks(range(6))
    ax.set_xticklabels(["attn-q","attn-k","attn-v","attn-proj","mlp-fc","mlp-proj"])
    ax.set_title(
        f"{title} [{tag}] (x={x_label}, iter={ITER_TAG})\n"
        f"d=dBIC=BIC(runner-up)-BIC(winner) (legend outside; never overlaps numbers)",
        fontsize=11
    )

    from matplotlib.patches import Patch
    cmap = plt.cm.viridis
    leg = [Patch(label="PL: CCDF ~ x^{-alpha}", facecolor=cmap(0.15)),
           Patch(label="LN: lognormal CCDF", facecolor=cmap(0.50)),
           Patch(label="TP: x^{-alpha} * exp(-(x/lambda)^beta)", facecolor=cmap(0.85))]
    ax.legend(handles=leg, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=True)
    plt.tight_layout()
    return fig

def fig_rep_grid(rep_groups, sel_by_group, x_label, tag):
    fig = plt.figure(figsize=(12.8, 7.4))
    rows, cols = 2, 3
    for i, gname in enumerate(rep_groups, 1):
        ax = plt.subplot(rows, cols, i)
        sel = sel_by_group[gname]
        g = sel["g"]; cc = sel["cc"]
        fit = sel["winner"]["fit"]
        cc_model = _ccdf_model_curve(g, fit)
        x_left, x_right, y_bot, y_top = _tail_zoom_limits(g, sel["fitmask"], sel["ccdf_max"])

        xs, ys = _subsample_xy(g, cc)
        ax.plot(xs, ys, linestyle="None", marker=".", markersize=DATA_MARKER_SIZE, alpha=DATA_ALPHA,
                markeredgewidth=DATA_EDGEWIDTH, label="data")
        ax.plot(g, cc_model, lw=MODEL_LINEWIDTH, alpha=0.95, label="model")

        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlim(x_left, x_right); ax.set_ylim(y_bot, y_top)
        ax.grid(True, which="both", ls="--", alpha=0.2)
        ttl = f"{gname}\n{_fmt_fit_short(fit, sel['tau'])}"
        if sel["ident"]:
            ttl += f"  [{sel['ident']}]"
        ax.set_title(ttl, fontsize=9)
        if i > 3: ax.set_xlabel("x")
        if i % 3 == 1: ax.set_ylabel("CCDF")
        ax.legend(fontsize=8, frameon=True, loc="lower left")

    plt.suptitle(
        f"Representative tail fits (points=data, line=model) [{tag}]\n"
        f"Selection: BIC on train bins + holdout SSE; dBIC=BIC(runner)-BIC(winner) | x={x_label}, iter={ITER_TAG}",
        y=0.98, fontsize=12
    )
    plt.tight_layout(rect=[0,0,1,0.92])
    return fig

def fig_appendix_ccdf(gname, sel, x_label, tag):
    g = sel["g"]; cc = sel["cc"]
    fit = sel["winner"]["fit"]
    cc_model = _ccdf_model_curve(g, fit)
    x_left, x_right, y_bot, y_top = _tail_zoom_limits(g, sel["fitmask"], sel["ccdf_max"])

    fig = plt.figure(figsize=(7.9, 4.9))
    ax = plt.gca()
    xs, ys = _subsample_xy(g, cc)
    ax.plot(xs, ys, linestyle="None", marker=".", markersize=DATA_MARKER_SIZE, alpha=DATA_ALPHA,
            markeredgewidth=DATA_EDGEWIDTH, label="data CCDF")
    ax.plot(g, cc_model, lw=MODEL_LINEWIDTH, label=f"model: {_fmt_fit_short(fit, sel['tau'])}")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(x_left, x_right); ax.set_ylim(y_bot, y_top)
    ax.grid(True, which="both", ls="--", alpha=0.25)
    ax.set_xlabel(f"x = {x_label}")
    ax.set_ylabel("CCDF(x)")
    ax.set_title(f"{gname} — tail CCDF [{tag}] (iter={ITER_TAG})", fontsize=11)

    strip = "Selection: BIC on train bins + holdout SSE. dBIC=BIC(runner)-BIC(winner). tau=(x_max/lambda)^beta (TP only)."
    ax.text(0.01, -0.28, strip, transform=ax.transAxes, fontsize=8, va="top")
    _legend_outside(ax)
    plt.tight_layout()
    return fig

def fig_appendix_beff(gname, sel, x_label, tag):
    g = sel["g"]; cc = sel["cc"]
    fit = sel["winner"]["fit"]
    beff = _beff_from_ccdf(g, cc, window=BEFF_WINDOW)
    bm = np.isfinite(beff) & sel["fitmask"]

    fig = plt.figure(figsize=(7.9, 4.7))
    ax = plt.gca()
    ax.plot(g[bm], beff[bm], lw=2, label="B_eff(data)")

    if fit["model"] == "powerlaw":
        ax.plot(g[bm], np.full_like(g[bm], fit["alpha"]), lw=2, label="PL: alpha (const)")
    elif fit["model"] == "tempered":
        ax.plot(g[bm], fit["alpha"] + fit["beta"] * (g[bm] / fit["lam"]) ** fit["beta"],
                lw=2, label="TP: alpha + beta*(x/lambda)^beta")
    else:
        lg = np.log(g[bm])
        ly = _log_ccdf_lognormal(g[bm], fit["mu"], fit["sigma"])
        d = np.gradient(ly, lg)
        ax.plot(g[bm], -d, lw=2, label="LN: numerical B_eff")

    ax.set_xscale("log")
    ax.grid(True, which="both", ls="--", alpha=0.25)
    ax.set_xlabel(f"x = {x_label}")
    ax.set_ylabel("B_eff(x) = - d log(CCDF) / d log(x)")
    ax.set_title(f"{gname} — B_eff [{tag}] (iter={ITER_TAG})", fontsize=11)

    box = _diag_box(sel)
    ax.text(1.02, 0.98, box, transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.95, edgecolor="0.6"))
    _legend_outside(ax)
    plt.tight_layout()
    return fig

def fig_beff_example(gname, sel, x_label, tag):
    g = sel["g"]; cc = sel["cc"]
    fit = sel["winner"]["fit"]
    beff = _beff_from_ccdf(g, cc, window=BEFF_WINDOW)
    bm = np.isfinite(beff) & sel["fitmask"]

    fig = plt.figure(figsize=(8.1, 4.7))
    ax = plt.gca()
    ax.plot(g[bm], beff[bm], lw=2, label="B_eff(data)")
    if fit["model"] == "powerlaw":
        ax.plot(g[bm], np.full_like(g[bm], fit["alpha"]), lw=2, label="PL: alpha (const)")
    elif fit["model"] == "tempered":
        ax.plot(g[bm], fit["alpha"] + fit["beta"] * (g[bm] / fit["lam"]) ** fit["beta"],
                lw=2, label="TP: alpha + beta*(x/lambda)^beta")
    else:
        lg = np.log(g[bm])
        ly = _log_ccdf_lognormal(g[bm], fit["mu"], fit["sigma"])
        d = np.gradient(ly, lg)
        ax.plot(g[bm], -d, lw=2, label="LN: numerical B_eff")

    ax.set_xscale("log")
    ax.grid(True, which="both", ls="--", alpha=0.25)
    ax.set_xlabel(f"x = {x_label}")
    ax.set_ylabel("B_eff(x) = - d log(CCDF) / d log(x)")
    ax.set_title(f"B_eff example: {gname} [{tag}] (iter={ITER_TAG})", fontsize=11)

    box = _diag_box(sel)
    ax.text(1.02, 0.98, box, transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.95, edgecolor="0.6"))
    _legend_outside(ax)
    plt.tight_layout()
    return fig

# -----------------------------
# Build outputs per map
# -----------------------------
MAPS = [
    ("noise", noise_map, "|g_a - g_b| (noise-only proxy)", "noise"),
    ("grad",  grad_map,  "|g| (stochastic gradient)",      "grad"),
    ("delta", delta_map, "|g_t - g_{t-1}| (rolling proxy)", "delta"),
]

def _pick_reps(summary_rows, n_total=MAIN_REP_N):
    reps = []
    for m in ["tempered", "lognormal", "powerlaw"]:
        cand = [r for r in summary_rows if r["model"] == m and np.isfinite(r["dBIC"])]
        cand.sort(key=lambda r: -r["dBIC"])
        reps.extend(cand[: max(1, n_total//3)])
    if len(reps) < n_total:
        allc = [r for r in summary_rows if np.isfinite(r["dBIC"])]
        allc.sort(key=lambda r: -r["dBIC"])
        seen = set([r["group"] for r in reps])
        for r in allc:
            if r["group"] not in seen:
                reps.append(r); seen.add(r["group"])
            if len(reps) >= n_total:
                break
    return reps[:n_total]

def build_for_map(name, data_map, x_label, tag):
    sel_by_group = {}
    summary = []
    for gname in ordered_groups():
        if gname not in data_map:
            continue
        sel = select_model_for_array(data_map[gname])
        if sel is None:
            continue
        sel_by_group[gname] = sel
        model = sel["winner"]["fit"]["model"]
        summary.append(dict(group=gname, model=model, dBIC=sel["dBIC"]))

    if len(summary) < 10:
        print(f"[warn] Too few fitted groups for {name}: {len(summary)}")
        return

    row_by = {r["group"]: r for r in summary}
    rows18 = [row_by[g] for g in ordered_groups() if g in row_by]
    if len(rows18) < 18:
        rows18 = rows18 + [rows18[-1]]*(18-len(rows18))

    # MAIN (only noise)
    if name == "noise" and MAKE_MAIN_NOISE:
        reps = _pick_reps(summary, MAIN_REP_N)
        rep_groups = [r["group"] for r in reps]

        f_map = fig_model_map(rows18, "Tail model selection across blocks", x_label, tag)
        f_rep = fig_rep_grid(rep_groups, sel_by_group, x_label, tag)

        main_map_path = out_dir / "main_noise_map_model_map.pdf"
        main_rep_path = out_dir / "main_noise_representative_fits.pdf"

        with PdfPages(str(main_map_path)) as pdf, _no_show():
            pdf.savefig(f_map, bbox_inches="tight")
        with PdfPages(str(main_rep_path)) as pdf, _no_show():
            pdf.savefig(f_rep, bbox_inches="tight")
        plt.close(f_map); plt.close(f_rep)

        if MAKE_MAIN_BEFF:
            # pick strong tempered else strongest overall
            cand = [(g, sel_by_group[g]["dBIC"], sel_by_group[g]["winner"]["fit"]["model"]) for g in sel_by_group.keys()]
            cand.sort(key=lambda t: (-(t[1] if np.isfinite(t[1]) else -1e9)))
            pick = next((g for g,d,m in cand if m=="tempered"), cand[0][0])
            f_beff = fig_beff_example(pick, sel_by_group[pick], x_label, tag)
            main_beff_path = out_dir / "main_noise_beff_example.pdf"
            with PdfPages(str(main_beff_path)) as pdf, _no_show():
                pdf.savefig(f_beff, bbox_inches="tight")
            plt.close(f_beff)

    # APPENDIX full audit PDF
    do_app = (name=="noise" and MAKE_APPENDIX_NOISE) or (name=="grad" and MAKE_APPENDIX_GRAD) or (name=="delta" and MAKE_APPENDIX_DELTA)
    if do_app:
        app_path = out_dir / f"appendix_{name}_full_audit.pdf"
        with PdfPages(str(app_path)) as pdf, _no_show():
            cover = fig_model_map(rows18, "Tail model selection across blocks", x_label, tag)
            pdf.savefig(cover, bbox_inches="tight")
            plt.close(cover)

            for gname in ordered_groups():
                if gname not in sel_by_group:
                    continue
                pdf.savefig(fig_appendix_ccdf(gname, sel_by_group[gname], x_label, tag), bbox_inches="tight")
                plt.close(plt.gcf())
                pdf.savefig(fig_appendix_beff(gname, sel_by_group[gname], x_label, tag), bbox_inches="tight")
                plt.close(plt.gcf())

    print(f"[ok] built: {name}")

for name, m, xlab, tag in MAPS:
    build_for_map(name, m, xlab, tag)

print("\n[done] All PDFs written to:", out_dir)
print(" - main_noise_map_model_map.pdf")
print(" - main_noise_representative_fits.pdf")
print(" - main_noise_beff_example.pdf (if enabled)")
print(" - appendix_noise_full_audit.pdf")
print(" - appendix_grad_full_audit.pdf")
print(" - appendix_delta_full_audit.pdf")

if not SHOW_PLOTS:
    plt.close("all")

[info] OUT_DIR: /home/coder/project/dcbench-c1-sophia/paper_figures_uai
[ok] built: noise
[ok] built: grad
[ok] built: delta

[done] All PDFs written to: /home/coder/project/dcbench-c1-sophia/paper_figures_uai
 - main_noise_map_model_map.pdf
 - main_noise_representative_fits.pdf
 - main_noise_beff_example.pdf (if enabled)
 - appendix_noise_full_audit.pdf
 - appendix_grad_full_audit.pdf
 - appendix_delta_full_audit.pdf
